# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is structured via a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. Upon loading, you will see a brief dataset description.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset URL (Croissant schema)
croissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load the dataset metadata from the Croissant schema
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset name and description (access via attributes, not dict)
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs via the Croissant metadata.

All entities are referenced by their Croissant `@id` fields. Listing all record sets and their fields with respective IDs:

In [ ]:
# List out the record sets and their fields, referencing by @id
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}")

for rs in record_sets:
    print(f"Record Set Name: {rs.name}")
    print(f"Record Set @id: {rs.id}")
    print("Fields:")
    for field in rs.fields:
        print(f" - {field.name} (@id={field.id}) [Type={field.data_type}]")
    print("Columns:")
    for col in rs.columns:
        print(f" - {col.name} (@id={col.id}) [Type={col.data_type}]")
    print("---")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame. All record sets and fields are referenced by their `@id` values, as listed above.
You may use these DataFrames for subsequent processing and visualization.

In [ ]:
# Extract data from all available record sets
dataframes = {}

for rs in record_sets:
    record_set_id = rs.id
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Record Set: '{rs.name}' (@id={record_set_id}) Columns:")
    print(df.columns.tolist())
    print(df.head())
    print("-")

## 4. Exploratory Data Analysis (EDA)
Let's process the record set containing hormone data, adrenal CT, and ultrasound results.
We'll reference this record set and its fields by their `@id`.

Common EDA tasks: Filter records for elevated hormone levels, normalize values, group by measurement type (if available).

In [ ]:
# Select a record set containing hormone data for EDA
hormone_rs = None
for rs in record_sets:
    if "hormone" in rs.name.lower() or "table 2" in rs.name.lower() or "laboratory" in rs.name.lower():
        hormone_rs = rs
        break

if hormone_rs:
    hormone_rs_id = hormone_rs.id
    df = dataframes[hormone_rs_id]
    print(f"Using Record Set: {hormone_rs.name} (@id={hormone_rs_id})")
    # Find numeric fields (such as hormone levels)
    numeric_fields = [field.id for field in hormone_rs.fields if field.data_type in ["Float", "Integer", "Number"]]
    print(f"Numeric Fields (@id): {numeric_fields}")
    
    # Choose first numeric field for demonstration (e.g., serum_17oh_progesterone @id)
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Example Numeric Field: {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        # Find a group field (e.g., imaging_type @id or measurement type)
        group_field_id = None
        for field in hormone_rs.fields:
            if "type" in field.name.lower() or "measurement" in field.name.lower():
                group_field_id = field.id
                break
        if group_field_id and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No numeric fields found in this record set.")
else:
    print("No suitable record set for hormone data found.")

## 5. Visualization
Visualize hormone values and distributions for the selected patients in the chosen record set.
The code uses `@id` references for axes and groupings.

In [ ]:
# Visualize hormone level distribution
if hormone_rs and numeric_fields:
    plt.figure(figsize=(8,5))
    df[numeric_field_id].dropna().plot.hist(bins=10, alpha=0.7)
    plt.title(f"Distribution of {numeric_field_id} in {hormone_rs.name}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    # Scatter by patient or group attribute if available
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,6))
        plt.scatter(df[group_field_id], df[numeric_field_id], c="blue", alpha=0.7)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library to load FAIR^2 dataset metadata, extract structured patient data via Croissant `@id` references, and perform basic filtering, normalization, and visualization.

**Key findings:**
- Data covers clinical features, hormone levels, imaging, and genetic variants for NC-21OHD in a rare, small-sample female cohort.
- High hormone values and imaging attributes can be filtered and normalized for exploratory analysis.
- Visualization sheds light on patient-level variation and data structure.

For further analysis, you may extend the notebook to deeper statistical modeling or annotation, following Croissant `@id` referencing throughout.